In [1]:
import pdfplumber
import pandas as pd
import uuid
from typing import List, Dict, Any


In [2]:
def bbox_intersects(a, b, min_overlap_ratio=0.01):
    """
    Return True if bbox a intersects bbox b.
    bboxes are (x0, top, x1, bottom).
    min_overlap_ratio: small tolerance to consider near-touching as intersection.
    """
    ax0, ay0, ax1, ay1 = a
    bx0, by0, bx1, by1 = b

    # intersection rectangle
    ix0, iy0 = max(ax0, bx0), max(ay0, by0)
    ix1, iy1 = min(ax1, bx1), min(ay1, by1)

    if ix1 <= ix0 or iy1 <= iy0:
        return False

    inter_area = (ix1 - ix0) * (iy1 - iy0)
    a_area = max(0, (ax1 - ax0) * (ay1 - ay0))
    b_area = max(0, (bx1 - bx0) * (by1 - by0))

    # consider it intersecting if any overlap exists or a small ratio threshold
    if inter_area <= 0:
        return False

    # overlap ratio relative to smaller area (helps catch nested small tables)
    smaller = min(a_area, b_area) if min(a_area, b_area) > 0 else 1
    return (inter_area / smaller) >= min_overlap_ratio

In [3]:
def merge_close_lines_to_text(words: List[Dict[str, Any]], tolerance: float = 3.0) -> str:
    """
    Group words by approximate top coordinate into lines and join them to produce paragraph text.
    `words` are the items returned by pdfplumber.page.extract_words()
    """
    # group by top coordinate with tolerance
    groups = {}
    for w in words:
        top = float(w.get("top", 0.0))
        # bucket key
        key = round(top / tolerance) * tolerance
        groups.setdefault(key, []).append((float(w.get("x0", 0.0)), w.get("text", "")))

    # sort lines top->bottom, words left->right
    ordered_keys = sorted(groups.keys())
    lines = []
    for k in ordered_keys:
        line_words = sorted(groups[k], key=lambda x: x[0])
        line_text = " ".join([t[1] for t in line_words])
        lines.append(line_text)
    return "\n".join(lines).strip()


In [4]:
def extract_tables_from_page(page) -> List[Dict]:
    """
    Extract table objects using pdfplumber's find_tables() when possible.
    Returns list of dicts: {table_index, table_id, bbox, rows(list of lists), df (if parsed)}
    """
    tables_out = []
    try:
        table_objs = page.find_tables()
    except Exception:
        table_objs = []

    # If pdfplumber found table objects (with bbox), use them
    for t_idx, t in enumerate(table_objs):
        tbbox = getattr(t, "bbox", None)  # (x0, top, x1, bottom)
        try:
            rows = t.extract()  # list of lists
        except Exception:
            # fallback to page.extract_tables() if extract fails
            try:
                rows = page.extract_tables()[t_idx]
            except Exception:
                rows = None
        df = None
        if rows:
            if len(rows) >= 2:
                df = pd.DataFrame(rows[1:], columns=rows[0])
            else:
                df = pd.DataFrame(rows)
        tables_out.append({
            "table_index": t_idx,
            "table_id": f"t_{t_idx}_{uuid.uuid4().hex[:8]}",
            "bbox": tuple(tbbox) if tbbox else None,
            "rows": rows,
            "df": df
        })

    # Fallback: if no table_objs but page.extract_tables() returns something, include them without bbox
    if not tables_out:
        try:
            ext_tables = page.extract_tables()
        except Exception:
            ext_tables = None
        if ext_tables:
            for t_idx, rows in enumerate(ext_tables):
                df = None
                if rows:
                    if len(rows) >= 2:
                        df = pd.DataFrame(rows[1:], columns=rows[0])
                    else:
                        df = pd.DataFrame(rows)
                tables_out.append({
                    "table_index": t_idx,
                    "table_id": f"t_{t_idx}_{uuid.uuid4().hex[:8]}",
                    "bbox": None,
                    "rows": rows,
                    "df": df
                })
    return tables_out


In [5]:
def detect_nested_tables(tables: List[Dict]) -> List[Dict]:
    """
    Given list of table dicts with bbox, detect parent-child relationships.
    If a table bbox fully contains another table bbox (with some margin), mark parent_table_id.
    Returns tables list augmented with 'parent_table_id' (or None).
    """
    # initialize
    for t in tables:
        t["parent_table_id"] = None

    # only consider bboxes that are not None
    with_bbox = [t for t in tables if t.get("bbox") is not None]

    for parent in with_bbox:
        p_bbox = parent["bbox"]
        for child in with_bbox:
            if child is parent:
                continue
            c_bbox = child["bbox"]
            # if parent bbox intersects child and is larger (likely contains)
            if bbox_intersects(p_bbox, c_bbox):
                # check containment (parent area significantly larger than child)
                px0, py0, px1, py1 = p_bbox
                cx0, cy0, cx1, cy1 = c_bbox
                p_area = max(0, (px1 - px0) * (py1 - py0))
                c_area = max(0, (cx1 - cx0) * (cy1 - cy0))
                if p_area > 2.5 * c_area:  # heuristic: parent area at least 2.5x child
                    # mark child parent if not already assigned (pick smallest parent)
                    # we choose the most immediate parent (smallest parent area that still contains)
                    current_parent = child.get("parent_table_id")
                    if not current_parent:
                        child["parent_table_id"] = parent["table_id"]
                    else:
                        # find current parent object to compare sizes
                        # (we assume table ids are unique and parent exists in tables list)
                        pass
    return tables

In [6]:
def extract_pdf_elements(pdf_path: str) -> List[Dict]:
    """
    Extract per-page elements:
      - narrative_text (text outside table bounding boxes)
      - tables: list with table metadata (table_id, parent_table_id, bbox, df, rows)
    Returns: list of pages with structure:
      {"page": int, "narrative_text": str, "tables": [ {table dict}, ... ]}
    """
    pages_output = []

    with pdfplumber.open(pdf_path) as pdf:
        for pidx, page in enumerate(pdf.pages, start=1):
            # 1) extract table objects (with bbox when possible)
            tables = extract_tables_from_page(page)

            # 2) detect nested tables (table inside table) and set parent_table_id where applicable
            if tables:
                tables = detect_nested_tables(tables)

            # 3) build list of table bboxes for filtering narrative words
            table_bboxes = [t["bbox"] for t in tables if t.get("bbox")]

            # 4) extract words and filter out words that fall into table bboxes (if bbox available)
            words = page.extract_words()  # list of dicts with x0, x1, top, bottom, text
            narrative_words = []
            for w in words:
                wbox = (float(w.get("x0", 0.0)), float(w.get("top", 0.0)),
                        float(w.get("x1", 0.0)), float(w.get("bottom", 0.0)))
                # if any table bbox intersects this word, it's considered part of a table
                in_any_table = False
                for tb in table_bboxes:
                    if bbox_intersects(wbox, tb):
                        in_any_table = True
                        break
                if not in_any_table:
                    narrative_words.append(w)

            # 5) reconstruct narrative text from words outside tables
            narrative_text = merge_close_lines_to_text(narrative_words)

            # 6) final packaging of table records: include textual fallback if df is None
            table_records = []
            for t in tables:
                # convert rows to a simple textual representation if df is None
                textual = None
                if t.get("rows"):
                    # join rows into a readable string fallback
                    textual = "\n".join([ " | ".join([str(cell) for cell in row]) for row in (t["rows"] or []) ])
                table_records.append({
                    "table_index": t.get("table_index"),
                    "table_id": t.get("table_id"),
                    "parent_table_id": t.get("parent_table_id"),
                    "bbox": t.get("bbox"),
                    "df": t.get("df"),        # pandas DataFrame or None
                    "rows": t.get("rows"),    # raw rows (list of lists) or None
                    "textual": textual        # fallback textual serialization
                })

            pages_output.append({
                "page": pidx,
                "narrative_text": narrative_text,
                "tables": table_records
            })

    return pages_output


In [7]:
pdf = extract_pdf_elements('assets/CP_Perjalanan Dinas.pdf')
pdf

[{'page': 1, 'narrative_text': 'PERJALANAN DINAS\nAUG 2025', 'tables': []},
 {'page': 2,
  'narrative_text': 'DEFINITION\nPERJALANAN DINAS adalah perjalanan yang dilakukan oleh Karyawan Perusahaan menuju lokasi di luar Lokasi Kerja Karyawan dan/atau kembali\nkeLokasiKerjaKaryawandalamrangkapelaksanaantugasataukepentinganPerusahaan.\nPERJALANAN DINAS JARAK DEKAT, disingkat sebagai PDJD, adalah Perjalanan Dinas dalam rangka pelaksanaan tugas atau kepentingan\nPerusahaan ke suatu wilayah dengan jarak antara Lokasi Kerja Karyawan dengan tujuan dinas <125 km (kurang dari seratus dua puluh lima\nkilometer)dalam1(satu)kaliperjalanan.\nPERJALANANDINASJARAKJAUH,disingkatsebagaiPDJJ,adalahPerjalananDinas dalamrangkapelaksanaantugasataukepentinganPerusahaan\nke suatu wilayah di dalam negeri maupun luar negeri negeri dengan jarak antara Lokasi Kerja Karyawan dengan tujuan dinas ≥125 km (lebih dari\natausamadenganseratusduapuluhlimakilometer)dalam1(satu)kaliperjalanandandapatmemilikiunsurmenginap.\

In [11]:
pdf[6]

{'page': 7,
 'narrative_text': 'PERJALANAN DINAS - KETENTUAN UMUM',
 'tables': [{'table_index': 0,
   'table_id': 't_0_41cd0923',
   'parent_table_id': None,
   'bbox': (24.544, 48.119333333333316, 695.46, 369.562),
   'df':                                       \
   0  PenggantianBiaya\nPerjalananDinas   
   1                  AlokasiBiayaDinas   
   
                      PERJALANAN DINAS/ BUSINESS TRAVEL  
   0  PDJJ Tidak\nJenis Biaya PDJD PDJD Menginap PDJ...  
   1  KetentuanPerjalananDinasbagiKaryawanyangmelaku...  ,
   'rows': [['', 'PERJALANAN DINAS/ BUSINESS TRAVEL'],
    ['PenggantianBiaya\nPerjalananDinas',
     'PDJJ Tidak\nJenis Biaya PDJD PDJD Menginap PDJJ\nMenginap\nUang Saku Dinas - Sesuai ketentuan pemberian Uang Saku -\nDinas\nUang Makan Dinas - - Sesuai ketentuan pemberian Uang\nMakan Dinas\nBiaya Transportasi Dinas SesuaiketentuanpemberianBiayaTransportasiDinas\nBiaya Akomodasi - Sesuai ketentuan pemberian Biaya -\nAkomodasi Dinas\nBiaya Lain-lain Diberikanjikadit

In [12]:
len(pdf[6]['tables'])

2

In [14]:
pdf[6]['tables'][0]['df']

,,PERJALANAN DINAS/ BUSINESS TRAVEL
0,PenggantianBiaya\nPerjalananDinas,PDJJ Tidak\nJenis Biaya PDJD PDJD Menginap PDJ...
1,AlokasiBiayaDinas,KetentuanPerjalananDinasbagiKaryawanyangmelaku...


In [15]:
pdf[6]['tables'][1]['df']

,Jenis Biaya,PDJD,PDJD Menginap,PDJJ,PDJJ Tidak\nMenginap
0,Uang Saku Dinas,-,Sesuai ketentuan pemberian Uang Saku\nDinas,None,-
1,Uang Makan Dinas,-,-,Sesuai ketentuan pemberian Uang\nMakan Dinas,None
2,Biaya Transportasi Dinas,SesuaiketentuanpemberianBiayaTransportasiDinas,None,None,None
3,Biaya Akomodasi,-,Sesuai ketentuan pemberian Biaya\nAkomodasi Dinas,None,-
4,Biaya Lain-lain,Diberikanjikaditetapkanoleh Perusahaan yang di...,None,None,None
5,Cash Advance,Tidak Diperbolehkan,None,Diperbolehkan,None


In [16]:
pdf[6]['tables'][1]

{'table_index': 1,
 'table_id': 't_1_ff2a1e4b',
 'parent_table_id': None,
 'bbox': (163.32, 78.47000000000003, 672.7, 292.47),
 'df':                 Jenis Biaya  \
 0           Uang Saku Dinas   
 1          Uang Makan Dinas   
 2  Biaya Transportasi Dinas   
 3           Biaya Akomodasi   
 4           Biaya Lain-lain   
 5              Cash Advance   
 
                                                 PDJD  \
 0                                                  -   
 1                                                  -   
 2     SesuaiketentuanpemberianBiayaTransportasiDinas   
 3                                                  -   
 4  Diberikanjikaditetapkanoleh Perusahaan yang di...   
 5                                Tidak Diperbolehkan   
 
                                        PDJD Menginap  \
 0        Sesuai ketentuan pemberian Uang Saku\nDinas   
 1                                                  -   
 2                                               None   
 3  Sesuai k

In [20]:
pdf[-2]['tables'][0]['df']

,Gol.,Uang Saku Dinas,None,None,Uang Makan Dinas,None,None,None,None,Kelas\nPenerbangan
0,None,Dalam Negeri\n(dalam ribuan\nRupiah),"Asia1),\nAfrika\n(dalamUSD)","Non Asia&\nAfrika,\nSingapore,\nHong Kong,\nda...",Dalam Negeri (dalam ribuan Rupiah),None,None,"Asia1), Afrika\n(dalam USD)","Non Asia&\nAfrika,\nSingapore,\nHong Kong, dan...",None
1,None,None,None,None,Makan Pagi,Makan\nSiang,Makan\nMalam,None,None,None
2,10,-,None,None,Sesuai Tagihan,None,None,None,None,Business
3,9,None,None,None,None,None,None,None,None,Economy2)
4,8,None,None,None,None,None,None,None,None,Economy
5,7,90,16,32,None,None,None,None,None,None
6,6,60,None,None,35,65,50,33,70,None
7,5,40,12,25,30,50,40,None,None,None
8,4,None,None,None,None,None,None,None,None,None
9,3,30,-,-,30,40,30,None,None,None


In [23]:
df = pdf[-2]['tables'][0]['df']
dct = df.to_dict(orient='records')
dct

C:\Users\USER\AppData\Local\Temp\ipykernel_5288\2067353714.py:2: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  dct = df.to_dict(orient='records')


[{'Gol.': None,
  'Uang Saku Dinas': 'Dalam Negeri\n(dalam ribuan\nRupiah)',
  None: 'Non Asia&\nAfrika,\nSingapore,\nHong Kong, dan\nJepang\n(dalam USD)',
  'Uang Makan Dinas': 'Dalam Negeri (dalam ribuan Rupiah)',
  'Kelas\nPenerbangan': None},
 {'Gol.': None,
  'Uang Saku Dinas': None,
  None: None,
  'Uang Makan Dinas': 'Makan Pagi',
  'Kelas\nPenerbangan': None},
 {'Gol.': '10',
  'Uang Saku Dinas': '-',
  None: None,
  'Uang Makan Dinas': 'Sesuai Tagihan',
  'Kelas\nPenerbangan': 'Business'},
 {'Gol.': '9',
  'Uang Saku Dinas': None,
  None: None,
  'Uang Makan Dinas': None,
  'Kelas\nPenerbangan': 'Economy2)'},
 {'Gol.': '8',
  'Uang Saku Dinas': None,
  None: None,
  'Uang Makan Dinas': None,
  'Kelas\nPenerbangan': 'Economy'},
 {'Gol.': '7',
  'Uang Saku Dinas': '90',
  None: None,
  'Uang Makan Dinas': None,
  'Kelas\nPenerbangan': None},
 {'Gol.': '6',
  'Uang Saku Dinas': '60',
  None: '70',
  'Uang Makan Dinas': '35',
  'Kelas\nPenerbangan': None},
 {'Gol.': '5',
  'Uang S